In [ ]:
import os
import glob
import kagglehub

# Define raw data destination
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

# Download dataset directly into output_dir
download_path = kagglehub.dataset_download(
    "mmumairkhattak/e-commerce-orders-dataset-2026-scra",
    output_dir=output_dir
)

# Locate the downloaded CSV file
csv_files = glob.glob(os.path.join(output_dir, "*.csv"))
target_raw_csv = csv_files[1]

print(f"Dataset successfully placed in: {target_raw_csv}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("ECommerce_Data_Ingestion")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")

In [ ]:
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Order_Date", StringType(), True),
    StructField("Customer_Age", IntegerType(), True),
    StructField("Customer_Gender", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Payment_Method", StringType(), True),
    StructField("Order_Amount", DoubleType(), True),
    StructField("Returned", StringType(), True),
])

df_raw = spark.read.option("header", "true").schema(schema).csv(target_raw_csv)

print(f"Raw record count: {df_raw.count()}")
df_raw.show(5)

In [ ]:
from pyspark.sql.functions import (
    coalesce,
    col,
    current_timestamp,
    lit,
    to_timestamp,
    when,
)

df_cleaned = (
    df_raw.withColumn(
        "Order_Timestamp", to_timestamp(col("Order_Date"), "yyyy-MM-dd")
    )
    .filter((col("Order_Amount").isNotNull()) & (col("Order_Amount") > 0.0))
    .withColumn("Payment_Method", coalesce(col("Payment_Method"), lit("Unknown")))
    .withColumn("Returned_Flag", when(col("Returned") == "Yes", 1).otherwise(0))
    .withColumn("Ingested_At", current_timestamp())
    .drop("Order_Date", "Returned")
)

df_cleaned.printSchema()
df_cleaned.show(5)

In [ ]:
curated_output_path = "../data/curated/cleaned_orders.parquet"

df_cleaned.write \
    .mode("overwrite") \
    .parquet(curated_output_path)

print(f"Cleaned dataset saved locally to: {curated_output_path}")